# Week 08 — Neural Network Surrogate Training

Same architecture family as W4-W6: MLP with H ∈ {16, 32}, four regularisation variants (plain / dropout / weight-decay / ensemble), 5-fold CV across 8 configs.

Re-trains on the latest data including W6 portal returns (16/16/21/36/26/26/36/46 pts).

F2/F4/F5/F6/F8 had new bests in W6 — fresh NN gradients at these new best points.

In [1]:
import sys, json
from pathlib import Path
import numpy as np
sys.path.append('../src')
import nn_models as nm

MODELS_DIR = Path('../models/week_08')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

WIDTHS = [16, 32]
VARIANTS = list(nm.VARIANTS)

def load(n):
    X = np.load(f'../data/function_{n}/initial_inputs.npy')
    Y = np.load(f'../data/function_{n}/initial_outputs.npy')
    return X, Y

In [2]:
def search_and_save(n, verbose=True):
    X, Y = load(n)
    baseline = float(Y.std())
    results = []
    for H in WIDTHS:
        for v in VARIANTS:
            rmse = nm.cv_rmse(X, Y, v, hidden=H, n_splits=5, seed=0)
            results.append((rmse, H, v))
    results.sort(key=lambda r: r[0])
    best_rmse, best_H, best_v = results[0]
    improvement = (baseline - best_rmse) / baseline * 100
    beat_baseline = best_rmse < baseline

    if verbose:
        print(f'F{n}: {X.shape[0]} pts, {X.shape[1]}D, baseline RMSE = {baseline:.4f}')
        print(f'  All configs (sorted):')
        for r, H, v in results:
            mark = ' ← BEST' if (r, H, v) == (best_rmse, best_H, best_v) else ''
            print(f'    H={H:3d}  {v:10s}  CV RMSE = {r:.4f}{mark}')
        status = '✓ beats baseline' if beat_baseline else '✗ WORSE than baseline'
        print(f'  → best: H={best_H}, {best_v}, RMSE={best_rmse:.4f} ({improvement:+.1f}% vs baseline) {status}')

    # Fit final model on all data
    models, meta = nm.fit_final(X, Y, best_v, best_H)
    meta['cv_rmse'] = best_rmse
    meta['baseline_rmse'] = baseline
    meta['beats_baseline'] = beat_baseline
    meta['all_configs'] = [{'hidden': H, 'variant': v, 'rmse': r} for r, H, v in results]

    # Gradient at current best point
    x_best = X[Y.argmax()]
    grad = nm.gradient_at(models, meta, x_best)
    meta['gradient_at_best'] = grad.tolist()
    meta['x_best'] = x_best.tolist()
    meta['y_best'] = float(Y.max())

    if verbose:
        print(f'  Gradient dY/dx at best point: {np.round(grad, 3).tolist()}')

    nm.save(models, meta, MODELS_DIR / f'function_{n}_nn.pt')
    return meta

## Train all 8 functions

In [3]:
all_meta = {}
for n in range(1, 9):
    print('=' * 72)
    all_meta[n] = search_and_save(n, verbose=True)
    print()

F1: 17 pts, 2D, baseline RMSE = 0.0017
  All configs (sorted):
    H= 32  dropout     CV RMSE = 0.0028 ← BEST
    H= 16  dropout     CV RMSE = 0.0028
    H= 32  ensemble    CV RMSE = 0.0029
    H= 16  ensemble    CV RMSE = 0.0041
    H= 32  wd          CV RMSE = 0.0043
    H= 32  plain       CV RMSE = 0.0045
    H= 16  wd          CV RMSE = 0.0046
    H= 16  plain       CV RMSE = 0.0048
  → best: H=32, dropout, RMSE=0.0028 (-60.3% vs baseline) ✗ WORSE than baseline
  Gradient dY/dx at best point: [0.023000000044703484, -0.003000000026077032]


F2: 17 pts, 2D, baseline RMSE = 0.2442
  All configs (sorted):
    H= 16  dropout     CV RMSE = 0.2126 ← BEST
    H= 32  dropout     CV RMSE = 0.2219
    H= 16  ensemble    CV RMSE = 0.2735
    H= 32  ensemble    CV RMSE = 0.2891
    H= 32  wd          CV RMSE = 0.2955
    H= 32  plain       CV RMSE = 0.3043
    H= 16  wd          CV RMSE = 0.3416
    H= 16  plain       CV RMSE = 0.3520
  → best: H=16, dropout, RMSE=0.2126 (+12.9% vs baseline) ✓ beats baseline
  Gradient dY/dx at best point: [0.9179999828338623, 0.8920000195503235]



F3: 22 pts, 3D, baseline RMSE = 0.0737
  All configs (sorted):
    H= 16  ensemble    CV RMSE = 0.0693 ← BEST
    H= 32  wd          CV RMSE = 0.0722
    H= 32  plain       CV RMSE = 0.0724
    H= 32  ensemble    CV RMSE = 0.0747
    H= 32  dropout     CV RMSE = 0.0816
    H= 16  wd          CV RMSE = 0.0820
    H= 16  dropout     CV RMSE = 0.0845
    H= 16  plain       CV RMSE = 0.0864
  → best: H=16, ensemble, RMSE=0.0693 (+6.0% vs baseline) ✓ beats baseline


  Gradient dY/dx at best point: [0.00800000037997961, 0.0010000000474974513, 0.375]



F4: 37 pts, 4D, baseline RMSE = 9.2928
  All configs (sorted):
    H= 32  plain       CV RMSE = 3.9922 ← BEST
    H= 32  wd          CV RMSE = 4.0653
    H= 32  ensemble    CV RMSE = 4.3149
    H= 16  wd          CV RMSE = 4.3368
    H= 16  ensemble    CV RMSE = 4.3467
    H= 16  plain       CV RMSE = 4.3515
    H= 32  dropout     CV RMSE = 5.1593
    H= 16  dropout     CV RMSE = 6.5318
  → best: H=32, plain, RMSE=3.9922 (+57.0% vs baseline) ✓ beats baseline
  Gradient dY/dx at best point: [-12.82800006866455, 10.092000007629395, -1.8930000066757202, 5.047999858856201]



...

F6: 27 pts, 5D, baseline RMSE = 0.6501
  All configs (sorted):
    H= 16  ensemble    CV RMSE = 0.3482 ← BEST
    H= 32  ensemble    CV RMSE = 0.3491
    H= 16  plain       CV RMSE = 0.3574
    H= 16  wd          CV RMSE = 0.3596
    H= 32  wd          CV RMSE = 0.3693
    H= 32  plain       CV RMSE = 0.3738
    H= 32  dropout     CV RMSE = 0.4119
    H= 16  dropout     CV RMSE = 0.4191
  → best: H=16, ensemble, RMSE=0.3482 (+46.4% vs baseline) ✓ beats baseline


  Gradient dY/dx at best point: [-0.5680000185966492, -0.5699999928474426, -1.2760000228881836, -1.3930000066757202, -2.0429999828338623]



F7: 37 pts, 6D, baseline RMSE = 0.5151
  All configs (sorted):
    H= 16  dropout     CV RMSE = 0.2955 ← BEST
    H= 32  dropout     CV RMSE = 0.3010
    H= 16  ensemble    CV RMSE = 0.3567
    H= 32  ensemble    CV RMSE = 0.4107
    H= 32  wd          CV RMSE = 0.4286
    H= 32  plain       CV RMSE = 0.4310
    H= 16  wd          CV RMSE = 0.4637
    H= 16  plain       CV RMSE = 0.4665
  → best: H=16, dropout, RMSE=0.2955 (+42.6% vs baseline) ✓ beats baseline
  Gradient dY/dx at best point: [-0.22300000488758087, 0.10199999809265137, -0.032999999821186066, -0.0949999988079071, -0.20800000429153442, 0.1940000057220459]


F8: 47 pts, 8D, baseline RMSE = 1.1372
  All configs (sorted):
    H= 32  ensemble    CV RMSE = 0.4503 ← BEST
    H= 16  dropout     CV RMSE = 0.4522
    H= 32  plain       CV RMSE = 0.4582
    H= 16  ensemble    CV RMSE = 0.4613
    H= 32  dropout     CV RMSE = 0.4729
    H= 32  wd          CV RMSE = 0.4741
    H= 16  wd          CV RMSE = 0.5110
    H= 16  plain       CV RMSE = 0.5212
  → best: H=32, ensemble, RMSE=0.4503 (+60.4% vs baseline) ✓ beats baseline


  Gradient dY/dx at best point: [-0.5490000247955322, 0.06199999898672104, 0.2849999964237213, 0.17000000178813934, 0.4560000002384186, 0.04600000008940697, -0.39500001072883606, 0.11299999803304672]



## F1 sign classifier

F1 is numerically ~0 for almost all points. Training an NN classifier on sign(y > 0) gives the analyze step a map of "where is the function positive" — useful for Q3/Q6 in the reflection.

In [4]:
X1, Y1 = load(1)
pos_frac = (Y1 > 0).mean()
print(f'F1: {len(Y1)} pts, {(Y1 > 0).sum()} positive, {(Y1 <= 0).sum()} zero-or-negative ({pos_frac:.0%} positive)')

if 0 < pos_frac < 1:
    clf, loo_acc = nm.train_sign_classifier(X1, Y1, hidden=16)
    nm.save_classifier(clf, loo_acc, d_in=X1.shape[1], hidden=16, path=MODELS_DIR / 'function_1_classifier.pt')
    print(f'Sign classifier trained. LOO accuracy = {loo_acc:.2%}')
else:
    print('Classifier skipped (all samples are one class).')

F1: 17 pts, 11 positive, 6 zero-or-negative (65% positive)


Sign classifier trained. LOO accuracy = 70.59%


## Summary

In [5]:
print(f"{'F':>2}  {'H':>3}  {'variant':10s}  {'CV RMSE':>10s}  {'baseline':>10s}  {'improve%':>8s}  {'beats?':>6s}")
print('-' * 62)
for n, m in all_meta.items():
    improve = (m['baseline_rmse'] - m['cv_rmse']) / m['baseline_rmse'] * 100
    mark = '✓' if m['beats_baseline'] else '✗'
    print(f'{n:>2}  {m["hidden"]:>3}  {m["variant"]:10s}  {m["cv_rmse"]:>10.4f}  {m["baseline_rmse"]:>10.4f}  {improve:>+7.1f}%  {mark:>6s}')

 F    H  variant        CV RMSE    baseline  improve%  beats?
--------------------------------------------------------------
 1   32  dropout         0.0028      0.0017    -60.3%       ✗
 2   16  dropout         0.2126      0.2442    +12.9%       ✓
 3   16  ensemble        0.0693      0.0737     +6.0%       ✓
 4   32  plain           3.9922      9.2928    +57.0%       ✓
 5   32  plain         105.2627    925.6395    +88.6%       ✓
 6   16  ensemble        0.3482      0.6501    +46.4%       ✓
 7   16  dropout         0.2955      0.5151    +42.6%       ✓
 8   32  ensemble        0.4503      1.1372    +60.4%       ✓
